# Buku Kerja Hari 2 — Functions, Modules & File Handling

Pendamping **Modul 2**. Estimasi 4–6 jam. Prasyarat: Hari 1.

**Tujuan:** menulis function yang rapi (parameter, default, `*args/**kwargs`, return, docstring), memakai lambda/map/filter/comprehension, membuat module sendiri, menangani error dengan `try/except`, dan membaca/menulis file teks & CSV dengan `pathlib`/`csv`.

| Bagian | Isi |
|---|---|
| A | Functions (L1–L8) |
| B | Lambda, map, filter, comprehension (L9–L13) |
| C | Modules & packages (L14–L15) |
| D | Error handling (L16–L18) |
| E | File handling: teks & CSV (L19–L20) |
| F | Mini Project 1 & 2 (P1, P2) |
| G | Rekap |

> Latihan yang menulis file memakai folder `latihan_hari2/` (dibuat otomatis di folder notebook).

> **Notebook SOLUSI** — semua latihan sudah terjawab dan dieksekusi; gunakan untuk membandingkan dengan jawaban Anda.

In [1]:
# @title Jalankan sel ini lebih dulu (alat bantu pemeriksa jawaban)
from contextlib import contextmanager
from unittest.mock import patch

STRICT = True
_hasil = {}

@contextmanager
def periksa(kode):
    """Dipakai di sel 'Periksa jawaban': gagal -> pesan petunjuk; lolos -> tanda centang."""
    try:
        yield
        _hasil[kode] = True
        print(f"✅ {kode} benar!")
    except AssertionError as e:
        _hasil[kode] = False
        print(f"❌ {kode} belum tepat: {e}")
        if STRICT: raise
    except Exception as e:
        _hasil[kode] = False
        print(f"❌ {kode} belum bisa dijalankan: {type(e).__name__}: {e}")
        if STRICT: raise

@contextmanager
def masukan(*jawaban):
    """Simulasi input(): with masukan('Ani', '80'): ... -> input() mengembalikan 'Ani' lalu '80'."""
    with patch("builtins.input", side_effect=list(jawaban)):
        yield

def ringkasan(semua):
    """Cetak rekap latihan: ✅ benar, ❌ salah, ⏳ belum dicoba."""
    benar = 0
    for kode in semua:
        s = _hasil.get(kode)
        tanda = "⏳ belum dicoba" if s is None else ("✅ benar" if s else "❌ perlu diperbaiki")
        benar += bool(s)
        print(f"{kode:<6} {tanda}")
    print(f"\nSkor: {benar} dari {len(semua)} latihan benar")

import sys, importlib
from pathlib import Path
WORK = Path("latihan_hari2")
WORK.mkdir(exist_ok=True)
if str(WORK.resolve()) not in sys.path:
    sys.path.insert(0, str(WORK.resolve()))


## A. Functions

**Ringkas:** function membungkus langkah yang bisa dipakai ulang (*DRY*). Pola: `def nama(parameter): ... return hasil`. Parameter bisa punya nilai default (`def f(a, b=10)`), `*args` menampung argumen posisi berlebih (tuple), `**kwargs` menampung argumen bernama berlebih (dict). Tulis **docstring** di baris pertama isi function.

In [2]:
def sapa(nama, salam="Halo"):
    """Mengembalikan kalimat sapaan."""
    return f"{salam}, {nama}!"

def jumlahkan(*angka):
    return sum(angka)

def tampil(**data):
    return [f"{k}: {v}" for k, v in data.items()]

print(sapa("Rina"), sapa("Rina", "Selamat pagi"))
print(jumlahkan(1, 2, 3, 4))
print(tampil(nama="Ani", ipk=3.8))

Halo, Rina! Selamat pagi, Rina!
10
['nama: Ani', 'ipk: 3.8']


### Latihan L1 — luas_segitiga

Buat function `luas_segitiga(alas, tinggi)` yang **mengembalikan** luas (`0.5 × alas × tinggi`).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
FUNCTION luas_segitiga(alas, tinggi)
    RETURN 0.5 × alas × tinggi
```

</details>

In [3]:
def luas_segitiga(alas, tinggi):
    return 0.5 * alas * tinggi

In [4]:
# Periksa jawaban
with periksa('L1'):
    assert luas_segitiga(10, 4) == 20, "luas_segitiga(10, 4) harus 20"
    assert luas_segitiga(3, 3) == 4.5
    assert luas_segitiga(0, 5) == 0

✅ L1 benar!


### Latihan L2 — cek_genap

Buat `cek_genap(n)` yang mengembalikan `True` jika `n` genap, `False` jika ganjil.

In [5]:
def cek_genap(n):
    return n % 2 == 0

In [6]:
# Periksa jawaban
with periksa('L2'):
    assert cek_genap(8) is True and cek_genap(7) is False
    assert cek_genap(0) is True

✅ L2 benar!


### Latihan L3 — rata_rata dengan *args

Buat `rata_rata(*angka)` yang mengembalikan rata-rata semua argumen; jika tidak ada argumen kembalikan `0`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
JIKA tidak ada angka → 0
SELAIN ITU → jumlah(angka) / banyak(angka)
```

</details>

In [7]:
def rata_rata(*angka):
    if not angka:
        return 0
    return sum(angka) / len(angka)

In [8]:
# Periksa jawaban
with periksa('L3'):
    assert rata_rata(80, 90, 70) == 80
    assert rata_rata(5) == 5
    assert rata_rata() == 0, "tanpa argumen harus 0 (hindari ZeroDivisionError)"

✅ L3 benar!


### Latihan L4 — info_mahasiswa dengan **kwargs

Buat `info_mahasiswa(**data)` yang mengembalikan **list string** `"kunci: nilai"` sesuai urutan argumen.

In [9]:
def info_mahasiswa(**data):
    return [f"{k}: {v}" for k, v in data.items()]

In [10]:
# Periksa jawaban
with periksa('L4'):
    assert info_mahasiswa(nama="Ani", ipk=3.8) == ["nama: Ani", "ipk: 3.8"]
    assert info_mahasiswa() == []

✅ L4 benar!


### Latihan L5 — konversi_nilai

Buat `konversi_nilai(angka)`: `A` (≥85), `B` (≥70), `C` (≥60), `D` (≥50), selain itu `E`. Nilai di luar 0–100 → kembalikan `"Tidak valid"` (gunakan *early return*).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
JIKA angka di luar 0..100 → "Tidak valid"
cek dari batas tertinggi ke terendah, return pada kecocokan pertama
```

</details>

In [11]:
def konversi_nilai(angka):
    if angka < 0 or angka > 100:
        return "Tidak valid"
    if angka >= 85: return "A"
    if angka >= 70: return "B"
    if angka >= 60: return "C"
    if angka >= 50: return "D"
    return "E"

In [12]:
# Periksa jawaban
with periksa('L5'):
    assert [konversi_nilai(n) for n in (95, 85, 75, 65, 55, 20)] == ["A", "A", "B", "C", "D", "E"]
    assert konversi_nilai(105) == "Tidak valid" and konversi_nilai(-1) == "Tidak valid"

✅ L5 benar!


### Latihan L6 — hitung_diskon dengan default

Buat `hitung_diskon(total, persen=10)`: jika `total >= 100000` kembalikan **jumlah diskon** (`total × persen / 100`), selain itu `0`.

In [13]:
def hitung_diskon(total, persen=10):
    if total >= 100000:
        return total * persen / 100
    return 0

In [14]:
# Periksa jawaban
with periksa('L6'):
    assert hitung_diskon(200000) == 20000, "default persen = 10"
    assert hitung_diskon(200000, 20) == 40000
    assert hitung_diskon(99999) == 0

✅ L6 benar!


### Latihan L7 — min_max (multiple return)

Buat `min_max(data)` yang mengembalikan **tuple** `(nilai_minimum, nilai_maksimum)`. Setelah itu, lakukan unpacking: `kecil, besar = min_max([3, 1, 4, 2])`.

In [15]:
def min_max(data):
    return min(data), max(data)

kecil, besar = min_max([3, 1, 4, 2])

In [16]:
# Periksa jawaban
with periksa('L7'):
    assert min_max([3, 1, 4, 2]) == (1, 4)
    assert (kecil, besar) == (1, 4)

✅ L7 benar!


### Latihan L8 — Docstring dan hitung_bmi

Buat `hitung_bmi(berat, tinggi)` (`berat / tinggi²`) **dengan docstring** yang tidak kosong. Kembalikan nilai dibulatkan 1 desimal.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
FUNCTION hitung_bmi(berat, tinggi)
    """deskripsi singkat"""
    RETURN round(berat / tinggi ** 2, 1)
```

</details>

In [17]:
def hitung_bmi(berat, tinggi):
    """Menghitung BMI = berat (kg) / tinggi (m) kuadrat, dibulatkan 1 desimal."""
    return round(berat / tinggi ** 2, 1)

In [18]:
# Periksa jawaban
with periksa('L8'):
    assert hitung_bmi(60, 1.7) == 20.8, f"hasil Anda: {hitung_bmi(60, 1.7)}"
    assert hitung_bmi.__doc__ and hitung_bmi.__doc__.strip(), "tambahkan docstring di baris pertama function"

✅ L8 benar!


## B. Lambda, map, filter, dan Comprehension

**Ringkas:** list comprehension `[ekspresi for x in data if kondisi]` adalah cara paling Pythonic membuat list baru. `lambda x: ...` = function satu baris tanpa nama, cocok sebagai `key=` di `sorted`. `map(f, data)` menerapkan f ke tiap elemen; `filter(f, data)` menyaring.

In [19]:
angka = [1, 2, 3, 4, 5]
print([x ** 2 for x in angka])
print([x for x in angka if x % 2 == 0])
print(list(map(lambda x: x * 2, angka)))
print(list(filter(lambda x: x > 3, angka)))
print(sorted(["pir", "apel", "kiwi"], key=lambda s: len(s)))
print({x: x ** 2 for x in range(1, 4)})

[1, 4, 9, 16, 25]
[2, 4]
[2, 4, 6, 8, 10]
[4, 5]
['pir', 'apel', 'kiwi']
{1: 1, 2: 4, 3: 9}


### Latihan L9 — Kuadrat 1–20

Dengan list comprehension, isi `kuadrat` dengan kuadrat bilangan 1 sampai 20.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
kuadrat ← [x ** 2 untuk x di range(1, 21)]
```

</details>

In [20]:
kuadrat = [x ** 2 for x in range(1, 21)]

In [21]:
# Periksa jawaban
with periksa('L9'):
    assert len(kuadrat) == 20 and kuadrat[0] == 1 and kuadrat[-1] == 400, f"kuadrat: {kuadrat[:3]}...{kuadrat[-1:]}"

✅ L9 benar!


### Latihan L10 — Bonus nilai lulus

Dari `nilai`, ambil hanya yang ≥ 75 lalu kalikan 1.1 dan **bulatkan 1 desimal**. Simpan ke `bonus`.

In [22]:
nilai = [60, 75, 40, 90, 55, 80]
bonus = [round(n * 1.1, 1) for n in nilai if n >= 75]

In [23]:
# Periksa jawaban
with periksa('L10'):
    assert bonus == [82.5, 99.0, 88.0], f"bonus: {bonus}"

✅ L10 benar!


### Latihan L11 — filter + lambda

Dengan `filter` dan `lambda`, ambil nama yang panjangnya **> 4** dari `nama` (hasil berupa **list**) dan simpan di `panjang`.

In [24]:
nama = ["Ani", "Budi", "Citra", "Eko", "Fajar"]
panjang = list(filter(lambda n: len(n) > 4, nama))

In [25]:
# Periksa jawaban
with periksa('L11'):
    assert panjang == ["Citra", "Fajar"], f"panjang: {panjang}"

✅ L11 benar!


### Latihan L12 — Dictionary comprehension

Buat `paritas`: dictionary `{angka: "genap" / "ganjil"}` untuk angka 1–10.

In [26]:
paritas = {x: ("genap" if x % 2 == 0 else "ganjil") for x in range(1, 11)}

In [27]:
# Periksa jawaban
with periksa('L12'):
    assert paritas[1] == "ganjil" and paritas[10] == "genap" and len(paritas) == 10

✅ L12 benar!


### Latihan L13 — Urutkan dengan lambda

Urutkan `mahasiswa` berdasarkan `ipk` **menurun** dengan `sorted(..., key=lambda ...)`; simpan di `urut`.

In [28]:
mahasiswa = [{"nama": "Ani", "ipk": 3.5}, {"nama": "Budi", "ipk": 3.9}, {"nama": "Citra", "ipk": 3.2}]
urut = sorted(mahasiswa, key=lambda m: m["ipk"], reverse=True)

In [29]:
# Periksa jawaban
with periksa('L13'):
    assert [m["nama"] for m in urut] == ["Budi", "Ani", "Citra"], f"urut: {urut}"

✅ L13 benar!


## C. Modules & Packages

**Ringkas:** module = file `.py` berisi function/variabel yang bisa di-`import`. Di notebook kita menulis file module dengan `Path.write_text(...)`, lalu meng-import-nya.

In [30]:
contoh = '''def kali(a, b):
    return a * b
'''
(WORK / "modul_contoh.py").write_text(contoh, encoding="utf-8")
import modul_contoh
print(modul_contoh.kali(6, 7))

42


### Latihan L14 — Module konversi.py

Isi string `isi_modul` dengan kode Python berisi tiga function: `celsius_ke_fahrenheit(c)`, `fahrenheit_ke_celsius(f)`, dan `meter_ke_km(m)`. Sel berikutnya menulisnya ke `konversi.py` lalu mengimpornya.

In [31]:
isi_modul = '''
def celsius_ke_fahrenheit(c):
    return c * 9 / 5 + 32

def fahrenheit_ke_celsius(f):
    return (f - 32) * 5 / 9

def meter_ke_km(m):
    return m / 1000
'''

In [32]:
# Periksa jawaban
with periksa('L14'):
    (WORK / "konversi.py").write_text(isi_modul, encoding="utf-8")
    sys.modules.pop("konversi", None); importlib.invalidate_caches()
    import konversi
    assert konversi.celsius_ke_fahrenheit(100) == 212
    assert round(konversi.fahrenheit_ke_celsius(212)) == 100
    assert konversi.meter_ke_km(2500) == 2.5

✅ L14 benar!


### Latihan L15 — Module string_utils.py

Sama seperti L14: isi `isi_modul2` dengan function `hitung_vokal(teks)`, `balik_string(teks)`, dan `apakah_palindrome(teks)` (abaikan spasi & huruf besar).

In [33]:
isi_modul2 = '''
def hitung_vokal(teks):
    return sum(1 for c in teks.lower() if c in "aiueo")

def balik_string(teks):
    return teks[::-1]

def apakah_palindrome(teks):
    s = "".join(teks.lower().split())
    return s == s[::-1]
'''

In [34]:
# Periksa jawaban
with periksa('L15'):
    (WORK / "string_utils.py").write_text(isi_modul2, encoding="utf-8")
    sys.modules.pop("string_utils", None); importlib.invalidate_caches()
    import string_utils as su
    assert su.hitung_vokal("Python Bootcamp") == 4
    assert su.balik_string("abc") == "cba"
    assert su.apakah_palindrome("Kasur ini rusak") is True and su.apakah_palindrome("Python") is False

✅ L15 benar!


## D. Error Handling

**Ringkas:** `try` menjalankan kode berisiko; `except Tipe` menangani error spesifik (hindari `except:` kosong); `else` jalan bila tidak ada error; `finally` selalu jalan; `raise` melempar error untuk validasi.

In [35]:
try:
    x = int("abc")
except ValueError as e:
    print("Gagal:", e)
else:
    print("ok", x)
finally:
    print("selesai")

Gagal: invalid literal for int() with base 10: 'abc'
selesai


### Latihan L16 — bagi_aman

Buat `bagi_aman(a, b)` yang mengembalikan `a / b`; jika `b == 0` (ZeroDivisionError) atau tipe tidak cocok (TypeError) kembalikan `None`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
COBA: RETURN a / b
TANGKAP ZeroDivisionError atau TypeError: RETURN None
```

</details>

In [36]:
def bagi_aman(a, b):
    try:
        return a / b
    except (ZeroDivisionError, TypeError):
        return None

In [37]:
# Periksa jawaban
with periksa('L16'):
    assert bagi_aman(10, 4) == 2.5
    assert bagi_aman(10, 0) is None, "pembagian nol harus None"
    assert bagi_aman(10, "a") is None, "tipe salah harus None"

✅ L16 benar!


### Latihan L17 — input_angka (ulangi sampai valid)

Buat `input_angka(prompt)` yang terus meminta input sampai bisa diubah ke `float`; jika bukan angka cetak `"Masukkan angka valid."` lalu ulangi. Kembalikan `float`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
ULANGI selamanya:
    COBA: RETURN float(input(prompt))
    TANGKAP ValueError: cetak peringatan
```

</details>

In [38]:
def input_angka(prompt):
    while True:
        try:
            return float(input(prompt))
        except ValueError:
            print("Masukkan angka valid.")

In [39]:
# Periksa jawaban
with periksa('L17'):
    with masukan("abc", "", "3.5"):
        hasil = input_angka("Angka: ")
    assert hasil == 3.5, f"hasil: {hasil!r}"
    with masukan("7"):
        assert input_angka("A: ") == 7.0

Masukkan angka valid.
Masukkan angka valid.
✅ L17 benar!


### Latihan L18 — Validasi dengan raise

Buat `set_ipk(ipk)` yang mengembalikan `ipk` jika 0–4, selain itu **melempar** `ValueError("IPK harus 0–4")`.

In [40]:
def set_ipk(ipk):
    if not 0 <= ipk <= 4:
        raise ValueError("IPK harus 0–4")
    return ipk

In [41]:
# Periksa jawaban
with periksa('L18'):
    assert set_ipk(3.5) == 3.5
    try:
        set_ipk(4.5)
    except ValueError as e:
        assert "0–4" in str(e) or "0-4" in str(e)
    else:
        raise AssertionError("set_ipk(4.5) seharusnya melempar ValueError")

✅ L18 benar!


## E. File Handling (Teks & CSV)

**Ringkas:** gunakan `with open(path, "w"/"r", encoding="utf-8")` agar file otomatis tertutup, atau `Path.write_text/read_text`. CSV: `csv.DictWriter/DictReader` dengan `newline=""` saat menulis.

In [42]:
with open(WORK / "demo.txt", "w", encoding="utf-8") as f:
    f.write("baris 1\nbaris 2\n")
print((WORK / "demo.txt").read_text(encoding="utf-8"))

baris 1
baris 2



### Latihan L19 — Simpan dan muat daftar (teks)

Buat `simpan_daftar(path, daftar)` (tulis satu item per baris) dan `muat_daftar(path)` (kembalikan list string tanpa newline; jika file **tidak ada** kembalikan `[]`).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
simpan: gabungkan item dengan "\n" lalu tulis
muat  : jika file tidak ada → []; selain itu baca dan splitlines()
```

</details>

In [43]:
def simpan_daftar(path, daftar):
    Path(path).write_text("\n".join(daftar) + "\n", encoding="utf-8")

def muat_daftar(path):
    p = Path(path)
    if not p.exists():
        return []
    return p.read_text(encoding="utf-8").splitlines()

In [44]:
# Periksa jawaban
with periksa('L19'):
    p = WORK / "mk.txt"
    simpan_daftar(p, ["Kalkulus", "Fisika", "AI"])
    assert muat_daftar(p) == ["Kalkulus", "Fisika", "AI"]
    assert muat_daftar(WORK / "tidak_ada.txt") == [], "file tidak ada → []"

✅ L19 benar!


### Latihan L20 — Simpan dan muat CSV

Buat `simpan_csv(path, rows)` (rows = list of dict; header dari key dict pertama) dan `muat_csv(path)` (kembalikan list of dict; file tidak ada → `[]`).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
simpan: buka file (newline="") → DictWriter(fieldnames) → writeheader → writerows
muat  : file tidak ada → []; selain itu list(DictReader(file))
```

</details>

In [45]:
import csv

def simpan_csv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

def muat_csv(path):
    p = Path(path)
    if not p.exists():
        return []
    with open(p, "r", encoding="utf-8") as f:
        return list(csv.DictReader(f))

In [46]:
# Periksa jawaban
with periksa('L20'):
    p = WORK / "nilai.csv"
    simpan_csv(p, [{"nama": "Ani", "nilai": 90}, {"nama": "Budi", "nilai": 85}])
    hasil = muat_csv(p)
    assert [r["nama"] for r in hasil] == ["Ani", "Budi"]
    assert hasil[0]["nilai"] == "90", "CSV selalu membaca sebagai teks (str)"
    assert muat_csv(WORK / "tidak_ada.csv") == []

✅ L20 benar!


## F. Mini Project

Mini project di sini bertumpu pada function L19–L20 (`simpan_csv`, `muat_csv`) sehingga data bertahan antar sesi.

### Latihan P1 — Mini Project 1 — Pencatat Kegiatan Akademik

Lengkapi function (data = list of dict, disimpan ke CSV):

- `tambah_kegiatan(data, tanggal, jenis, deskripsi, status="belum")` → tambah dict `{"tanggal","jenis","deskripsi","status"}`; `jenis` harus salah satu `tugas/kuliah/ujian`, jika tidak kembalikan `False`; sukses → `True`.
- `filter_kegiatan(data, kunci)` → list kegiatan yang `jenis` **atau** `status`-nya sama dengan `kunci`.
- `tandai_selesai(data, indeks)` → ubah `status` jadi `"selesai"` (indeks mulai 1); indeks tidak valid → `False`.
- `simpan_kegiatan(path, data)` dan `muat_kegiatan(path)` memakai `simpan_csv` / `muat_csv`.

In [47]:
JENIS_VALID = ("tugas", "kuliah", "ujian")

def tambah_kegiatan(data, tanggal, jenis, deskripsi, status="belum"):
    if jenis not in JENIS_VALID:
        return False
    data.append({"tanggal": tanggal, "jenis": jenis, "deskripsi": deskripsi, "status": status})
    return True

def filter_kegiatan(data, kunci):
    return [k for k in data if kunci in (k["jenis"], k["status"])]

def tandai_selesai(data, indeks):
    if not 1 <= indeks <= len(data):
        return False
    data[indeks - 1]["status"] = "selesai"
    return True

def simpan_kegiatan(path, data):
    simpan_csv(path, data)

def muat_kegiatan(path):
    return muat_csv(path)

In [48]:
# Periksa jawaban
with periksa('P1'):
    d = []
    assert tambah_kegiatan(d, "2026-10-01", "tugas", "Laporan Python") is True
    assert tambah_kegiatan(d, "2026-10-02", "kuliah", "Pemrograman", "selesai") is True
    assert tambah_kegiatan(d, "2026-10-03", "rapat", "Salah jenis") is False, "jenis tidak valid harus ditolak"
    assert len(d) == 2
    assert [k["deskripsi"] for k in filter_kegiatan(d, "tugas")] == ["Laporan Python"]
    assert [k["deskripsi"] for k in filter_kegiatan(d, "selesai")] == ["Pemrograman"]
    assert tandai_selesai(d, 1) is True and d[0]["status"] == "selesai"
    assert tandai_selesai(d, 9) is False
    p = WORK / "kegiatan.csv"
    simpan_kegiatan(p, d)
    assert muat_kegiatan(p) == d, "data hasil muat harus sama dengan data yang disimpan"
    assert muat_kegiatan(WORK / "belum_ada.csv") == []

✅ P1 benar!

### Latihan P2 — Mini Project 2 — Perpustakaan Mini

Data buku: list of dict `{"judul","penulis","tahun","status"}` (`status` = `"tersedia"` / `"dipinjam"`). Lengkapi:

- `tambah_buku(buku, judul, penulis, tahun)` → status awal `"tersedia"`.
- `pinjam(buku, judul)` → ubah ke `"dipinjam"`, kembalikan `True`; gagal (tidak ada / sudah dipinjam) → `False`. Pencocokan judul **tidak peka huruf besar/kecil**.
- `kembalikan(buku, judul)` → ubah ke `"tersedia"`; judul tidak ada → `False`.
- `cari(buku, kata)` → list buku yang `judul` **atau** `penulis`-nya mengandung `kata` (tidak peka huruf besar/kecil).

In [49]:
def tambah_buku(buku, judul, penulis, tahun):
    buku.append({"judul": judul, "penulis": penulis, "tahun": tahun, "status": "tersedia"})

def pinjam(buku, judul):
    for b in buku:
        if b["judul"].lower() == judul.lower() and b["status"] == "tersedia":
            b["status"] = "dipinjam"
            return True
    return False

def kembalikan(buku, judul):
    for b in buku:
        if b["judul"].lower() == judul.lower():
            b["status"] = "tersedia"
            return True
    return False

def cari(buku, kata):
    k = kata.lower()
    return [b for b in buku if k in b["judul"].lower() or k in b["penulis"].lower()]

In [50]:
# Periksa jawaban
with periksa('P2'):
    b = []
    tambah_buku(b, "Python Dasar", "Ani", 2024)
    tambah_buku(b, "Basis Data", "Budi", 2023)
    assert [x["status"] for x in b] == ["tersedia", "tersedia"]
    assert pinjam(b, "python dasar") is True, "pencocokan judul tidak peka huruf besar/kecil"
    assert pinjam(b, "Python Dasar") is False, "sudah dipinjam → False"
    assert pinjam(b, "Tidak Ada") is False
    assert b[0]["status"] == "dipinjam"
    assert [x["judul"] for x in cari(b, "budi")] == ["Basis Data"]
    assert [x["judul"] for x in cari(b, "DATA")] == ["Basis Data"]
    assert kembalikan(b, "PYTHON DASAR") is True and b[0]["status"] == "tersedia"
    assert kembalikan(b, "Tidak Ada") is False

✅ P2 benar!


## G. Rekap

1. Apa beda `return` dan `print` di dalam function? Mengapa function yang `return` lebih mudah diuji?
2. Kapan memakai `except ValueError` dibanding `except Exception`? Mengapa `except:` kosong berbahaya?
3. Satu kesalahan yang Anda temui hari ini (mis. `UnboundLocalError`, `FileNotFoundError`) dan cara memperbaikinya:

> *Jawaban Anda:* …

**Checklist sebelum Hari 3**

- [ ] Semua latihan L1–L20 berstatus ✅
- [ ] Mini Project 1 (P1) dan 2 (P2) lulus pemeriksaan
- [ ] Saya bisa menulis function dengan default, `*args`, `**kwargs`, dan docstring
- [ ] Saya bisa membuat module sendiri dan meng-import-nya
- [ ] Saya bisa membaca/menulis CSV tanpa baris kosong ganda

## Rekap hasil latihan

Jalankan sel di bawah setelah mengerjakan latihan.

In [51]:
ringkasan(['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8', 'L9', 'L10', 'L11', 'L12', 'L13', 'L14', 'L15', 'L16', 'L17', 'L18', 'L19', 'L20', 'P1', 'P2'])

L1     ✅ benar
L2     ✅ benar
L3     ✅ benar
L4     ✅ benar
L5     ✅ benar
L6     ✅ benar
L7     ✅ benar
L8     ✅ benar
L9     ✅ benar
L10    ✅ benar
L11    ✅ benar
L12    ✅ benar
L13    ✅ benar
L14    ✅ benar
L15    ✅ benar
L16    ✅ benar
L17    ✅ benar
L18    ✅ benar
L19    ✅ benar
L20    ✅ benar
P1     ✅ benar
P2     ✅ benar

Skor: 22 dari 22 latihan benar
